# Immune integration: Hierarchical annotation transfer

Transfer cell type annotations from annotated to unannotated cells using Leiden clustering
on the trained RNA latent space, followed by hierarchical majority-vote labeling.

**Strategy**: Broad clusters → adaptive sub-clustering → hierarchical labeling via majority vote.
All cells (annotated + unannotated) clustered together. Target ~150 final sub-clusters.

**Prerequisites**: NB5 (trained RNA model with latent space, kNN graph, UMAP)
**Input**: `results/immune_integration_rna/model/outputs/` (X_scVI, X_umap, distances)
**Output**: `results/immune_integration/annotation_transfer.csv`

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
from matplotlib import rcParams
import scipy
import os

rcParams["pdf.fonttype"] = 42

## Parameters

In [ ]:
# Papermill parameters
results_folder = "results/immune_integration/"
rna_model_folder = "results/immune_integration_rna/model/"
k_neighbors = 50
broad_resolution = 1.0
purity_threshold = 0.80
target_n_subclusters = 150

## Load data + latent space

In [ ]:
from data_loading_utils import load_atac_qc_metrics, load_scrublet_scores

# Load RNA adata (for obs metadata + annotations)
rna_path = os.path.join(results_folder, "adata_rna.h5ad")
adata = sc.read_h5ad(rna_path)
print(f"Loaded: {adata.shape}")

# Load scrublet and ATAC QC using safe reindex utilities
scrublet_path = os.path.join(results_folder, "scrublet_results.csv")
adata = load_scrublet_scores(adata, scrublet_path)

atac_qc_path = os.path.join(results_folder, "atac_qc_metrics.csv")
if os.path.exists(atac_qc_path):
    adata = load_atac_qc_metrics(adata, atac_qc_path)

In [ ]:
# Apply same cell filtering as NB5
if "counts" in adata.layers:
    adata.X = adata.layers["counts"]

# Use QC columns from NB1 (total_counts, n_genes already present)
if "total_counts" not in adata.obs.columns:
    total_counts = np.array(adata.X.sum(1)).squeeze()
    adata.obs["total_counts"] = total_counts
if "n_genes" not in adata.obs.columns:
    n_genes = np.array((adata.X > 0).sum(1)).squeeze()
    adata.obs["n_genes"] = n_genes

# Strict QC thresholds (matching NB5)
required_cols = ["n_genes", "total_counts", "mt_frac", "doublet_score", "total_fragments"]
missing = [c for c in required_cols if c not in adata.obs.columns]
if missing:
    raise ValueError(f"Missing required QC columns: {missing}")

mask = (
    (adata.obs["n_genes"] > 800)
    & (adata.obs["total_counts"] > 1500)
    & (adata.obs["total_counts"] < 80000)
    & (adata.obs["total_fragments"] > 2500)
    & (adata.obs["total_fragments"] < 80000)
    & (adata.obs["n_genes"] < 10000)
    & (adata.obs["mt_frac"] < 0.20)
    & (adata.obs["doublet_score"] < 0.20)
)
n_before = adata.n_obs
adata = adata[mask, :].copy()
print(f"Filtered {n_before:,} -> {adata.n_obs:,} cells ({n_before - adata.n_obs:,} removed)")

# Drop batches with too few cells
min_cells_per_batch = 100
batch_counts = adata.obs["batch"].value_counts()
small_batches = batch_counts[batch_counts < min_cells_per_batch].index.tolist()
if small_batches:
    n_before = adata.n_obs
    adata = adata[~adata.obs["batch"].isin(small_batches)].copy()
    print(f"Removed {len(small_batches)} batches with <{min_cells_per_batch} cells")
    print(f"  {n_before:,} -> {adata.n_obs:,} cells")

In [ ]:
# Load latent representation and UMAP from NB5
output_dir = os.path.join(rna_model_folder, "outputs")

X_scVI = pd.read_csv(os.path.join(output_dir, "X_scVI.csv"), index_col=0)
X_umap = pd.read_csv(os.path.join(output_dir, f"X_umap_k{k_neighbors}.csv"), index_col=0)

# Align to filtered adata
common_cells = adata.obs_names.intersection(X_scVI.index)
adata = adata[common_cells, :].copy()
adata.obsm["X_scVI"] = X_scVI.loc[common_cells].values
adata.obsm["X_umap"] = X_umap.loc[common_cells].values

print(f"Aligned cells: {adata.n_obs}")
print(f"Latent dim: {adata.obsm['X_scVI'].shape[1]}")

# Load kNN graph
distances = scipy.sparse.load_npz(os.path.join(output_dir, f"distances_euclidean_k{k_neighbors}.npz"))
connectivities = scipy.sparse.load_npz(os.path.join(output_dir, f"connectivities_euclidean_k{k_neighbors}.npz"))

# Subset graph to filtered cells (indices may have changed)
cell_idx = np.where(X_scVI.index.isin(common_cells))[0]
distances = distances[cell_idx][:, cell_idx]
connectivities = connectivities[cell_idx][:, cell_idx]

adata.obsp["distances"] = distances
adata.obsp["connectivities"] = connectivities
adata.uns["neighbors"] = {
    "connectivities_key": "connectivities",
    "distances_key": "distances",
    "params": {"n_neighbors": k_neighbors, "method": "umap", "metric": "euclidean"},
}

print(f"kNN graph loaded: {distances.shape}")

## Load annotation hierarchy

In [ ]:
# Parse annotation_hierarchy.md into lookup table

hierarchy_path = "docs/notebooks/immune_integration/annotation_hierarchy.md"
hierarchy_rows = []

with open(hierarchy_path) as f:
    for line in f:
        line = line.strip()
        if not line.startswith("|") or line.startswith("| ---") or line.startswith("| harmonized"):
            continue
        parts = [p.strip() for p in line.split("|")[1:-1]]
        if len(parts) < 5:
            continue
        name, l1, l2, l3, l4 = parts[:5]
        # Skip section headers (bold text)
        if name.startswith("**"):
            continue
        hierarchy_rows.append(
            {
                "harmonized_name": name,
                "level_1": l1,
                "level_2": l2,
                "level_3": l3,
                "level_4": l4,
            }
        )

hierarchy_df = pd.DataFrame(hierarchy_rows)
print(f"Hierarchy: {len(hierarchy_df)} cell types")
print(f"level_1 categories: {hierarchy_df['level_1'].nunique()}")
print(f"level_2 categories: {hierarchy_df['level_2'].nunique()}")
print(f"level_3 categories: {hierarchy_df['level_3'].nunique()}")
print(f"level_4 categories: {hierarchy_df['level_4'].nunique()}")

# Create lookup dicts
name_to_level1 = dict(zip(hierarchy_df["harmonized_name"], hierarchy_df["level_1"], strict=True))
name_to_level2 = dict(zip(hierarchy_df["harmonized_name"], hierarchy_df["level_2"], strict=True))
name_to_level3 = dict(zip(hierarchy_df["harmonized_name"], hierarchy_df["level_3"], strict=True))
name_to_level4 = dict(zip(hierarchy_df["harmonized_name"], hierarchy_df["level_4"], strict=True))

# Also build level_1 -> level_2/3/4 lookups
l1_to_l2 = dict(zip(hierarchy_df["level_1"], hierarchy_df["level_2"], strict=False))
l1_to_l3 = dict(zip(hierarchy_df["level_1"], hierarchy_df["level_3"], strict=False))
l1_to_l4 = dict(zip(hierarchy_df["level_1"], hierarchy_df["level_4"], strict=False))

In [ ]:
# Annotation status
n_annotated = adata.obs["harmonized_annotation"].notna().sum()
n_total = adata.n_obs
print(f"Annotated: {n_annotated}/{n_total} ({100 * n_annotated / n_total:.1f}%)")
print(f"Unannotated: {n_total - n_annotated}")

# Check for level_1: prefix annotations
if n_annotated > 0:
    annots = adata.obs["harmonized_annotation"].dropna()
    level1_prefix = annots.str.startswith("level_1:")
    n_coarse = level1_prefix.sum()
    n_fine = (~level1_prefix).sum()
    print(f"Fine annotations: {n_fine}")
    print(f"Coarse (level_1:) annotations: {n_coarse}")

# Per-dataset annotation counts
print("\nPer-dataset annotation status:")
for ds in sorted(adata.obs["dataset"].unique()):
    mask = adata.obs["dataset"] == ds
    n = mask.sum()
    n_ann = adata.obs.loc[mask, "harmonized_annotation"].notna().sum()
    print(f"  {ds}: {n_ann}/{n} ({100 * n_ann / n:.1f}%)")

## Step 1: Broad clustering

Leiden at `broad_resolution` on the kNN graph to get ~15-30 broad clusters.

In [ ]:
sc.tl.leiden(adata, resolution=broad_resolution, flavor="igraph", key_added="leiden_broad")

n_broad = adata.obs["leiden_broad"].nunique()
print(f"Broad clusters: {n_broad}")
print("Cluster sizes:")
print(adata.obs["leiden_broad"].value_counts().describe())

In [ ]:
palette = sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy

rcParams["figure.figsize"] = 7, 7
sc.pl.umap(adata, color="leiden_broad", palette=palette, size=1, legend_fontsize=8)

## Step 2: Adaptive sub-clustering

For each broad cluster, sub-cluster with resolution scaled to cluster size.
Larger clusters get higher resolution to avoid under-splitting.
Check annotation purity per sub-cluster.

In [ ]:
def adaptive_subcluster(adata, broad_key="leiden_broad", target_total=150, purity_thresh=0.80):
    """Adaptively sub-cluster each broad cluster.

    Resolution scales with cluster size. Re-clusters impure sub-clusters.
    """
    n_broad = adata.obs[broad_key].nunique()
    target_per_broad = max(2, target_total // n_broad)

    all_subclusters = pd.Series("", index=adata.obs_names, dtype=str)

    for broad_id in sorted(adata.obs[broad_key].unique(), key=int):
        mask = adata.obs[broad_key] == broad_id
        n_cells = mask.sum()

        if n_cells < 50:
            # Too small to sub-cluster
            all_subclusters.loc[mask] = f"{broad_id}_0"
            continue

        # Scale resolution with cluster size
        base_res = max(0.5, target_per_broad * (n_cells / 10000))
        sub_adata = adata[mask].copy()

        # Need neighbors for sub-clustering
        sc.pp.neighbors(sub_adata, use_rep="X_scVI", n_neighbors=min(30, n_cells - 1))
        sc.tl.leiden(sub_adata, resolution=base_res, flavor="igraph", key_added="leiden_sub")

        n_sub = sub_adata.obs["leiden_sub"].nunique()

        # Check purity of each sub-cluster
        for sub_id in sub_adata.obs["leiden_sub"].unique():
            sub_mask = sub_adata.obs["leiden_sub"] == sub_id
            sub_cells = sub_adata.obs_names[sub_mask]
            annots = adata.obs.loc[sub_cells, "harmonized_annotation"].dropna()

            if len(annots) > 0:
                _ = annots.value_counts().iloc[0] / len(annots)  # purity check

            # If impure and large enough, could re-cluster (optional refinement)
            cluster_label = f"{broad_id}_{sub_id}"
            all_subclusters.loc[sub_cells] = cluster_label

        print(f"Broad {broad_id}: {n_cells} cells -> {n_sub} sub-clusters (res={base_res:.2f})")

    return all_subclusters


adata.obs["cluster_id"] = adaptive_subcluster(
    adata,
    target_total=target_n_subclusters,
    purity_thresh=purity_threshold,
)

n_clusters = adata.obs["cluster_id"].nunique()
print(f"\nTotal sub-clusters: {n_clusters}")

## Step 3: Purity analysis

In [ ]:
def compute_purity(adata, cluster_key="cluster_id", annot_key="harmonized_annotation"):
    """Compute annotation purity per cluster."""
    rows = []
    for cid in sorted(adata.obs[cluster_key].unique()):
        mask = adata.obs[cluster_key] == cid
        n_cells = mask.sum()
        annots = adata.obs.loc[mask, annot_key].dropna()
        n_annotated = len(annots)

        if n_annotated > 0:
            top_annot = annots.value_counts().index[0]
            top_count = annots.value_counts().iloc[0]
            purity = top_count / n_annotated
            top3 = annots.value_counts().head(3).to_dict()
        else:
            top_annot = "unannotated"
            purity = np.nan
            top3 = {}

        rows.append(
            {
                "cluster_id": cid,
                "n_cells": n_cells,
                "n_annotated": n_annotated,
                "purity": purity,
                "top_annotation": top_annot,
                "top3_annotations": str(top3),
            }
        )
    return pd.DataFrame(rows)


purity_df = compute_purity(adata)

print(f"Clusters with purity >= {purity_threshold}: {(purity_df['purity'] >= purity_threshold).sum()}/{len(purity_df)}")
print(f"Clusters with purity < {purity_threshold}: {(purity_df['purity'] < purity_threshold).sum()}/{len(purity_df)}")
print(f"Unannotated clusters (no annotated cells): {purity_df['purity'].isna().sum()}")
print("\nPurity distribution:")
print(purity_df["purity"].describe())

In [ ]:
# Show low-purity clusters for manual review
low_purity = purity_df[purity_df["purity"] < purity_threshold].sort_values("purity")
if len(low_purity) > 0:
    print(f"Low-purity clusters ({len(low_purity)}):")
    for _, row in low_purity.iterrows():
        print(f"  {row['cluster_id']}: {row['n_cells']} cells, purity={row['purity']:.2f}, top={row['top_annotation']}")
        print(f"    top3: {row['top3_annotations']}")
else:
    print("All clusters above purity threshold!")

## Step 4: Hierarchical labeling

**First pass: level_2** (cell type families: T-cell, B-cell, Monocyte, NK, DC, etc.)
- Majority vote from annotated cells within each sub-cluster
- `level_1:` prefixed annotations contribute to level_2/3 voting but NOT finer levels

**Second pass: level_1** (finest) within each level_2 group
- Among sub-clusters labeled as e.g. "T-cell lineage", assign specific type
- Use only cells with full harmonized_annotation (not level_1:-prefixed) for voting

**Auto-populate**: level_3, level_4 from level_2 via hierarchy lookup

In [ ]:
def get_annotation_levels(annot_series):
    """Map harmonized_annotation values to hierarchy levels.

    Handles both fine annotations (direct lookup) and level_1: prefix annotations
    (used for level_2/3/4 voting only).
    """
    results = {
        "level_1": pd.Series(np.nan, index=annot_series.index),
        "level_2": pd.Series(np.nan, index=annot_series.index),
        "level_3": pd.Series(np.nan, index=annot_series.index),
        "level_4": pd.Series(np.nan, index=annot_series.index),
        "is_coarse": pd.Series(False, index=annot_series.index),
    }

    for idx, annot in annot_series.items():
        if pd.isna(annot):
            continue

        if annot.startswith("level_1:"):
            # Coarse annotation — extract level_1 category
            l1_val = annot.replace("level_1:", "").strip()
            results["is_coarse"].loc[idx] = True
            results["level_1"].loc[idx] = l1_val
            # Look up higher levels
            if l1_val in l1_to_l2:
                results["level_2"].loc[idx] = l1_to_l2[l1_val]
            if l1_val in l1_to_l3:
                results["level_3"].loc[idx] = l1_to_l3[l1_val]
            if l1_val in l1_to_l4:
                results["level_4"].loc[idx] = l1_to_l4[l1_val]
        else:
            # Fine annotation — direct lookup
            if annot in name_to_level1:
                results["level_1"].loc[idx] = name_to_level1[annot]
                results["level_2"].loc[idx] = name_to_level2[annot]
                results["level_3"].loc[idx] = name_to_level3[annot]
                results["level_4"].loc[idx] = name_to_level4[annot]

    return results


# Compute levels for all annotated cells
annots = adata.obs["harmonized_annotation"]
levels = get_annotation_levels(annots)
for key in ["level_1", "level_2", "level_3", "level_4", "is_coarse"]:
    adata.obs[f"source_{key}"] = levels[key]

print(f"Fine annotations: {(~adata.obs['source_is_coarse'] & annots.notna()).sum()}")
print(f"Coarse annotations: {adata.obs['source_is_coarse'].sum()}")

In [ ]:
def hierarchical_labeling(adata, purity_df, cluster_key="cluster_id"):
    """Assign hierarchical labels to clusters via majority vote.

    Pass 1: Assign level_2 using all annotated cells (fine + coarse).
    Pass 2: Assign finest harmonized_annotation within each level_2 group,
            using only fine annotations (not level_1: prefixed).
    Pass 3: Auto-populate level_3, level_4 from level_2 lookup.
    """
    # Initialize output columns
    adata.obs["transferred_level_2"] = np.nan
    adata.obs["transferred_level_1"] = np.nan
    adata.obs["transferred_level_3"] = np.nan
    adata.obs["transferred_level_4"] = np.nan
    adata.obs["transferred_harmonized_annotation"] = np.nan

    cluster_ids = sorted(adata.obs[cluster_key].unique())

    # --- Pass 1: level_2 by majority vote ---
    cluster_to_l2 = {}
    for cid in cluster_ids:
        mask = adata.obs[cluster_key] == cid
        l2_vals = adata.obs.loc[mask, "source_level_2"].dropna()
        if len(l2_vals) > 0:
            cluster_to_l2[cid] = l2_vals.value_counts().index[0]
        else:
            cluster_to_l2[cid] = "Unknown"

    for cid in cluster_ids:
        mask = adata.obs[cluster_key] == cid
        adata.obs.loc[mask, "transferred_level_2"] = cluster_to_l2[cid]

    # --- Pass 2: finest annotation within each level_2 group ---
    # Track how many clusters get each label (to add _cid suffix for duplicates)
    label_counts = {}

    for cid in cluster_ids:
        mask = adata.obs[cluster_key] == cid
        # Use only fine annotations (not level_1: prefixed)
        fine_mask = mask & (~adata.obs["source_is_coarse"])
        annots = adata.obs.loc[fine_mask, "harmonized_annotation"].dropna()

        if len(annots) > 0:
            top_label = annots.value_counts().index[0]
        else:
            # Fall back to level_2
            top_label = cluster_to_l2[cid]

        # Track for deduplication
        if top_label not in label_counts:
            label_counts[top_label] = []
        label_counts[top_label].append(cid)

        adata.obs.loc[mask, "transferred_harmonized_annotation"] = top_label

    # Add _cid suffix for clusters sharing the same majority label
    for label, cids in label_counts.items():
        if len(cids) > 1:
            for cid in cids:
                mask = adata.obs[cluster_key] == cid
                adata.obs.loc[mask, "transferred_harmonized_annotation"] = f"{label}_{cid}"

    # --- Pass 3: Auto-populate level_1, level_3, level_4 ---
    for cid in cluster_ids:
        mask = adata.obs[cluster_key] == cid
        l2 = cluster_to_l2[cid]

        # level_1 from majority vote of source_level_1
        l1_vals = adata.obs.loc[mask, "source_level_1"].dropna()
        if len(l1_vals) > 0:
            adata.obs.loc[mask, "transferred_level_1"] = l1_vals.value_counts().index[0]
        else:
            adata.obs.loc[mask, "transferred_level_1"] = l2

        # level_3 and level_4 from level_2 lookup
        l2_rows = hierarchy_df[hierarchy_df["level_2"] == l2]
        if len(l2_rows) > 0:
            adata.obs.loc[mask, "transferred_level_3"] = l2_rows["level_3"].iloc[0]
            adata.obs.loc[mask, "transferred_level_4"] = l2_rows["level_4"].iloc[0]

    return adata


adata = hierarchical_labeling(adata, purity_df)

print("Transferred annotations:")
print(f"  level_1: {adata.obs['transferred_level_1'].nunique()} categories")
print(f"  level_2: {adata.obs['transferred_level_2'].nunique()} categories")
print(f"  level_3: {adata.obs['transferred_level_3'].nunique()} categories")
print(f"  level_4: {adata.obs['transferred_level_4'].nunique()} categories")
print(f"  harmonized: {adata.obs['transferred_harmonized_annotation'].nunique()} categories")

## Visualization

In [ ]:
rcParams["figure.figsize"] = 7, 7
color_cols = [
    "transferred_level_2",
    "transferred_level_1",
    "transferred_harmonized_annotation",
    "dataset",
]
sc.pl.umap(
    adata,
    color=color_cols,
    ncols=1,
    palette=palette,
    size=1,
    legend_fontsize=8,
)

In [ ]:
# Compare original vs transferred for annotated cells
annotated_mask = adata.obs["harmonized_annotation"].notna() & ~adata.obs["source_is_coarse"]
if annotated_mask.sum() > 0:
    orig = adata.obs.loc[annotated_mask, "harmonized_annotation"]
    transferred = adata.obs.loc[annotated_mask, "transferred_harmonized_annotation"]
    # Strip _cid suffixes for comparison
    transferred_clean = transferred.str.replace(r"_\d+_\d+$", "", regex=True)
    agreement = (orig == transferred_clean).mean()
    print(f"Agreement (annotated cells): {agreement:.3f} ({(orig == transferred_clean).sum()}/{len(orig)})")

## Save

In [ ]:
# Save annotation transfer results
output_cols = [
    "leiden_broad",
    "cluster_id",
    "transferred_harmonized_annotation",
    "transferred_level_1",
    "transferred_level_2",
    "transferred_level_3",
    "transferred_level_4",
]
transfer_df = adata.obs[output_cols].copy()
output_path = os.path.join(results_folder, "annotation_transfer.csv")
transfer_df.to_csv(output_path)
print(f"Saved: {output_path}")
print(f"  {len(transfer_df)} cells, {transfer_df['cluster_id'].nunique()} clusters")

# Save purity summary for manual review
purity_path = os.path.join(results_folder, "annotation_transfer_purity.csv")
purity_df.to_csv(purity_path, index=False)
print(f"Saved purity summary: {purity_path}")